
# Transformers para Visión

La arquitectura del tranformer se propuso inicialmente para el aprendizaje de secuencia a secuencia, como la traducción automática. Con gran eficacia, los tranformers se convirtieron posteriormente en el modelo de elección en diversas tareas de procesamiento del lenguaje natural.
Sin embargo, en el campo de la visión artificial la arquitectura dominante
se había basado en las CNN. *¿Podemos adaptar tranformers para modelar datos de imagen*? Esta pregunta ha despertado un gran interés en la comunidad de visión artificial. Un [paper del 2020](https://arxiv.org/pdf/1911.03584.pdf) demostró teóricamente que la autoatención puede aprender a comportarse de manera similar a la convolución. Empíricamente, se tomaron parches de $2 \times 2$ de las imágenes como entrada, pero el pequeño tamaño del parche hace que el modelo solo sea aplicable a datos de imágenes con resoluciones bajas.

Sin restricciones específicas sobre el tamaño del parche, los *tranformers de visión* (ViT) extraen parches de las imágenes y los introducen en un encoder de tranformer para obtener una representación global, que finalmente se transformará para la clasificación. En particular, los tranformers muestran una mejor escalabilidad que las CNN: cuando se entrenan modelos más grandes en conjuntos de datos más grandes, los tranformers de visión superan a los ResNet por un margen significativo. Similar al panorama del diseño de arquitectura de red en el procesamiento del lenguaje natural, los tranformers también cambiaron las reglas del juego en la visión por computadora.




## Modelo



La siguiente figura representa la arquitectura modelo de los tranformers de visión. Esta arquitectura consta de una base que parchea las imágenes, un cuerpo basado en el encoder de un tranformer multicapa y una cabeza que transforma la representación global en la etiqueta de salida.

![Imgur](https://i.imgur.com/4qOAWmXl.png)




Considere una imagen de entrada con altura $h$, ancho $w$ y $c$ canales. Especificando la altura y el ancho del parche como $p$,
la imagen se divide en una secuencia de $m = hw/p^2$ parches,
donde cada parche se aplana a un vector de longitud $cp^2$.
De esta forma, los parches de imagen pueden ser tratados de manera similar a los tokens en secuencias de texto por encoders de tranformers. Un token especial “&lt;cls&gt;” (clasificación) y los $m$ parches de imagen aplanados se proyectan linealmente en una secuencia de vectores $m+1$, sumados con embeddings posicionales que se pueden aprender. El encoder de tranformer multicapa transforma los vectores de entrada $m+1$ en la misma cantidad de representaciones de vectores de salida de la misma longitud. Funciona exactamente de la misma manera que el encoder del tranformer original, solo que difiere en la posición de normalización. Dado que el token “&lt;cls&gt;”  atiende a todos los parches de imagen a través de la autoatención su representación desde la salida del encoder del tranformer
se transformará en la etiqueta de salida.

In [13]:
import torch
from torch import nn

## Patch Embedding

Para implementar un transformer de visión, comencemos con los embeddings de los parches. Dividir una imagen en parches y proyectar linealmente estos parches aplanados se puede simplificar como una sola operación de convolución, donde tanto el tamaño del kernel como el tamaño del stride se establecen en el tamaño del parche.


In [ ]:
# Módulo PatchEmbedding: divide una imagen en "parches" y los convierte en vectores de embeddings
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=96, patch_size=16, num_hiddens=512):
        super().__init__()
        # Define el tamaño total de la imagen y el de cada parche (alto, ancho)
        img_size, patch_size = (img_size, img_size), (patch_size, patch_size)

        # Calcula cuántos parches se pueden obtener de la imagen
        # Ejemplo: una imagen 96x96 con parches 16x16 → (96/16)*(96/16)=36 parches
        self.num_patches = (img_size[0] // patch_size[0]) * (
            img_size[1] // patch_size[1]
        )

        # Capa convolucional que actúa como extractor de parches
        # - kernel_size y stride iguales al tamaño del parche ⇒ no se superponen
        # - num_hiddens: tamaño del embedding de cada parche
        # - LazyConv2d infiere automáticamente el número de canales de entrada
        self.conv = nn.LazyConv2d(num_hiddens, kernel_size=patch_size,
                                  stride=patch_size)

    def forward(self, X):
        # Aplica la convolución para obtener los parches
        # El resultado tiene forma (batch_size, num_hiddens, n_h, n_w)
        out = self.conv(X)

        # Reorganiza el tensor:
        # 1️⃣ flatten(2): aplana las dimensiones espaciales (n_h * n_w = número de parches)
        # 2️⃣ transpose(1, 2): intercambia canales y parches → (batch_size, num_patches, num_hiddens)
        return out.flatten(2).transpose(1, 2)


En el siguiente ejemplo, tomando imágenes con una altura y un ancho de `img_size` como entrada, se generan `(img_size//patch_size)**2` parches que se proyectan linealmente en vectores de longitud `num_hiddens`.


In [ ]:
# Función auxiliar para verificar que un tensor tenga la forma esperada
def check_shape(a, shape):
    """Lanza un error si el tensor 'a' no tiene la forma 'shape' esperada."""
    assert a.shape == shape, \
        f"tensor's shape {a.shape} != expected shape {shape}"

# Parámetros base: tamaño de imagen, tamaño de parche, dimensión del embedding y tamaño del lote
img_size, patch_size, num_hiddens, batch_size = 96, 16, 512, 4

# Creamos una instancia de PatchEmbedding
patch_emb = PatchEmbedding(img_size, patch_size, num_hiddens)

# Generamos un lote de 4 imágenes RGB de 96x96 píxeles
X = torch.randn(batch_size, 3, img_size, img_size)

# Verificamos que la salida tenga la forma correcta:
# (batch_size, número_de_patches, dimensión_del_embedding)
# Para 96x96 con parches de 16x16 → (96/16)^2 = 36 parches
check_shape(
    patch_emb(X),
    (batch_size, (img_size // patch_size) ** 2, num_hiddens)
)


## Encoder del Transformer de Vision

El MLP del encoder transformer de visión es ligeramente diferente del a la red feed forward posicional del encoder del transformer original. Primero, aquí la función de activación usa la unidad lineal de error gaussiano (GELU), que puede considerarse como una versión más suave de ReLU. En segundo lugar, el dropout se aplica a la salida de cada capa densa en el MLP para la regularización.

In [ ]:
# Módulo MLP (Multilayer Perceptron) usado dentro del bloque Transformer del Vision Transformer (ViT)
class ViTMLP(nn.Module):
    def __init__(self, mlp_num_hiddens, mlp_num_outputs, dropout=0.5):
        super().__init__()
        # Capa lineal que expande la dimensionalidad de la entrada
        self.dense1 = nn.LazyLinear(mlp_num_hiddens)
        # Activación GELU (usada en Transformers por su suavidad y rendimiento)
        self.gelu = nn.GELU()
        # Dropout para regularización y evitar sobreajuste
        self.dropout1 = nn.Dropout(dropout)
        # Capa lineal que proyecta de vuelta al tamaño original
        self.dense2 = nn.LazyLinear(mlp_num_outputs)
        # Segundo dropout después de la proyección final
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x):
        # Flujo: Linear → GELU → Dropout → Linear → Dropout
        # Aplica las dos capas lineales con activación intermedia y dropout
        return self.dropout2(
            self.dense2(
                self.dropout1(
                    self.gelu(self.dense1(x))
                )
            )
        )


La implementación del bloque del encoder del transformer de visión simplemente sigue el diseño de prenormalización, donde la normalización se aplica justo *antes* de la atención multiples cabezales o el MLP. A diferencia de la posnormalización, donde la normalización se coloca justo *después* de las conexiones residuales, la prenormalización conduce a un entrenamiento más efectivo o eficiente para los transformers.


In [ ]:
# Capa de Atención Multi-Cabeza (Multi-Head Attention)
# Implementa la parte central del Transformer: múltiples cabezas de atención en paralelo.
class MultiHeadAttentionLayer(nn.Module):
    def __init__(self, hid_dim, n_heads, dropout, device):
        super().__init__()

        # Asegura que la dimensión total sea divisible entre el número de cabezas
        assert hid_dim % n_heads == 0

        # Dimensiones básicas
        self.hid_dim = hid_dim          # Dimensión total del embedding
        self.n_heads = n_heads          # Número de cabezas de atención
        self.head_dim = hid_dim // n_heads  # Dimensión por cabeza

        # Proyecciones lineales para query (Q), key (K) y value (V)
        self.fc_q = nn.Linear(hid_dim, hid_dim)
        self.fc_k = nn.Linear(hid_dim, hid_dim)
        self.fc_v = nn.Linear(hid_dim, hid_dim)

        # Capa final que combina todas las cabezas
        self.fc_o = nn.Linear(hid_dim, hid_dim)

        # Dropout para regularización
        self.dropout = nn.Dropout(dropout)

        # Escala para normalizar los puntajes de atención
        self.scale = torch.sqrt(torch.FloatTensor([self.head_dim])).to(device)

    def forward(self, query, key, value, mask=None):
        batch_size = query.shape[0]

        # query, key, value tienen forma: [batch size, seq len, hid dim]

        # Proyecciones lineales
        Q = self.fc_q(query)
        K = self.fc_k(key)
        V = self.fc_v(value)

        # Reorganizamos Q, K, V para separar las múltiples cabezas
        # De (batch, seq_len, hid_dim) → (batch, n_heads, seq_len, head_dim)
        Q = Q.view(batch_size, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        K = K.view(batch_size, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        V = V.view(batch_size, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)

        # Calculamos la "energía" o puntajes de atención: Q * K^T / sqrt(d_k)
        energy = torch.matmul(Q, K.permute(0, 1, 3, 2)) / self.scale
        # energy = [batch size, n_heads, query len, key len]

        # Aplicamos máscara si existe (por ejemplo, para ignorar padding)
        if mask is not None:
            energy = energy.masked_fill(mask == 0, -1e10)

        # Calculamos los pesos de atención (softmax sobre la dimensión key_len)
        attention = torch.softmax(energy, dim=-1)
        # attention = [batch size, n_heads, query len, key len]

        # Aplicamos la atención a los valores V
        x = torch.matmul(self.dropout(attention), V)
        # x = [batch size, n_heads, query len, head dim]

        # Reordenamos las dimensiones para recombinar todas las cabezas
        x = x.permute(0, 2, 1, 3).contiguous()
        # x = [batch size, query len, n_heads, head dim]

        # Aplanamos las cabezas → concatenamos todas (n_heads * head_dim = hid_dim)
        x = x.view(batch_size, -1, self.hid_dim)
        # x = [batch size, query len, hid dim]

        # Pasamos por la capa lineal final que mezcla la información de todas las cabezas
        x = self.fc_o(x)
        # x = [batch size, query len, hid dim]

        # Devolvemos la salida final y los pesos de atención
        return x, attention


In [ ]:
# Bloque básico del Vision Transformer (ViT)
# Combina atención multi-cabeza + MLP con normalización y conexiones residuales
class ViTBlock(nn.Module):
    def __init__(self, num_hiddens, norm_shape, mlp_num_hiddens,
                 num_heads, dropout, device):
        super().__init__()

        # 1️⃣ Normalización por capas antes de la atención (pre-norm)
        self.ln1 = nn.LayerNorm(norm_shape).to(device)

        # 2️⃣ Capa de atención multi-cabeza
        self.attention = MultiHeadAttentionLayer(num_hiddens, num_heads,
                                                 dropout, device).to(device)

        # 3️⃣ Segunda normalización por capas antes del MLP
        self.ln2 = nn.LayerNorm(norm_shape).to(device)

        # 4️⃣ Bloque MLP (feed-forward network)
        self.mlp = ViTMLP(mlp_num_hiddens, num_hiddens, dropout).to(device)

    def forward(self, X, mask=None):
        # 🔹 Normaliza las entradas antes de la atención
        X = self.ln1(X)

        # 🔹 Calcula la atención auto-regresiva (usa X como query, key y value)
        att_output, _ = self.attention(X, X, X, mask)

        # 🔹 Agrega conexión residual entre entrada y salida de atención
        # y aplica normalización antes del MLP
        return X + self.mlp(self.ln2(X + att_output))


Igual que en el transformer original, cualquier bloque de encoder de transformer de visión no cambia su forma de entrada.


In [17]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X = torch.ones((2, 100, 24)).to(device)
encoder_blk = ViTBlock(24, 24, 48, 8, 0.5, device)
encoder_blk.eval()
check_shape(encoder_blk(X), X.shape)

## Juntar todo

El paso hacia adelante de los transformers de visión es sencillo. Primero, las imágenes de entrada se introducen en una instancia `PatchEmbedding`, cuya salida se concatena con el embedding del token “&lt;cls&gt;”. Se suman los embeddings posicionales aprendibles antes del dropout. Luego, la salida se alimenta al encoder del transformer que apila las instancias `num_blks` de la clase `ViTBlock`. Finalmente, la representación del token “&lt;cls&gt;” token es proyectado por la cabeza de la red.


In [ ]:
# Implementación completa de un Vision Transformer (ViT)
class ViT(nn.Module):
    """Vision Transformer."""
    def __init__(self, img_size, patch_size, num_hiddens, mlp_num_hiddens,
                 num_heads, num_blks, emb_dropout, blk_dropout, device, lr=0.1,
                 use_bias=False, num_classes=10):
        super().__init__()

        # 🔹 Convierte la imagen en una secuencia de embeddings de parches
        self.patch_embedding = PatchEmbedding(img_size, patch_size, num_hiddens)

        # 🔹 Token de clasificación [CLS]: representa el resumen global de la imagen
        self.cls_token = nn.Parameter(torch.zeros(1, 1, num_hiddens))

        # 🔹 Número total de pasos (parches + token de clasificación)
        num_steps = self.patch_embedding.num_patches + 1

        # 🔹 Embeddings posicionales aprendibles (indican la posición de cada parche)
        self.pos_embedding = nn.Parameter(torch.randn(1, num_steps, num_hiddens))

        # 🔹 Dropout aplicado a los embeddings (para regularización)
        self.dropout = nn.Dropout(emb_dropout)

        # 🔹 Secuencia de bloques Transformer
        self.blks = nn.Sequential()
        for i in range(num_blks):
            self.blks.add_module(f"{i}", ViTBlock(
                num_hiddens, num_hiddens, mlp_num_hiddens,
                num_heads, blk_dropout, device))

        # 🔹 Capa de salida (clasificador final)
        # - Normaliza la salida del [CLS] token
        # - Pasa por una capa lineal para predecir la clase
        self.head = nn.Sequential(
            nn.LayerNorm(num_hiddens),
            nn.Linear(num_hiddens, num_classes)
        )

    def forward(self, X):
        # 1️⃣ Convierte la imagen en embeddings de parches
        X = self.patch_embedding(X)

        # 2️⃣ Añade el token [CLS] al inicio de la secuencia
        X = torch.cat((self.cls_token.expand(X.shape[0], -1, -1), X), 1)

        # 3️⃣ Suma los embeddings posicionales y aplica dropout
        X = self.dropout(X + self.pos_embedding)

        # 4️⃣ Pasa por todos los bloques Transformer
        for blk in self.blks:
            X = blk(X)

        # 5️⃣ Toma la representación del token [CLS] y la pasa al clasificador
        return self.head(X[:, 0])


## Training

Training a vision transformer on the Fashion-MNIST dataset is just like how CNNs were trained in :numref:`chap_modern_cnn`.


In [ ]:
# Implementa un Vision Transformer (ViT) completo: convierte una imagen en parches,
# procesa cada parche como un "token" y usa bloques Transformer para clasificar.
class ViT(nn.Module):
    def __init__(self, img_size, patch_size, num_hiddens, mlp_num_hiddens,
                 num_heads, num_blks, emb_dropout, blk_dropout, device, lr=0.1,
                 use_bias=False, num_classes=10):
        super().__init__()

        # Convierte imagen → secuencia de embeddings de parches
        self.patch_embedding = PatchEmbedding(img_size, patch_size, num_hiddens)

        # Token especial [CLS] que resume la imagen completa
        self.cls_token = nn.Parameter(torch.zeros(1, 1, num_hiddens))

        # Cantidad total de tokens (parches + 1 token [CLS])
        num_steps = self.patch_embedding.num_patches + 1

        # Embeddings posicionales aprendibles para indicar la posición de cada parche
        self.pos_embedding = nn.Parameter(torch.randn(1, num_steps, num_hiddens))

        # Dropout sobre embeddings
        self.dropout = nn.Dropout(emb_dropout)

        # Secuencia de bloques Transformer (atención + MLP)
        self.blks = nn.Sequential()
        for i in range(num_blks):
            self.blks.add_module(f"{i}", ViTBlock(
                num_hiddens, num_hiddens, mlp_num_hiddens,
                num_heads, blk_dropout, device))

        # Capa final: normaliza y clasifica
        self.head = nn.Sequential(
            nn.LayerNorm(num_hiddens),
            nn.Linear(num_hiddens, num_classes)
        )

    def forward(self, X):
        # 1️⃣ Convierte la imagen a embeddings de parches
        X = self.patch_embedding(X)
        # 2️⃣ Añade el token [CLS] al inicio de la secuencia
        X = torch.cat((self.cls_token.expand(X.shape[0], -1, -1), X), 1)
        # 3️⃣ Suma embeddings posicionales y aplica dropout
        X = self.dropout(X + self.pos_embedding)
        # 4️⃣ Procesa con los bloques Transformer
        for blk in self.blks:
            X = blk(X)
        # 5️⃣ Usa el token [CLS] para la predicción final
        return self.head(X[:, 0])


In [ ]:
# Función para entrenar un clasificador (como ViT o CNN) sobre el dataset Fashion-MNIST
def train_FashionMNIST_classifier(model, lr, num_epochs, resize=None):
  batch_size = 128
  # Usa GPU si está disponible, sino CPU
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  model = model.to(device)

  # Función de pérdida: entropía cruzada (típica para clasificación)
  loss = nn.CrossEntropyLoss(reduction='none')

  # Optimizador Adam con tasa de aprendizaje lr
  trainer = torch.optim.Adam(model.parameters(), lr=lr)

  # Carga los datos de entrenamiento y prueba
  # resize permite ajustar el tamaño de las imágenes (por ej. a 96x96 para ViT)
  train_iter, test_iter = load_data_fashion_mnist(batch_size, resize=resize)

  # 🔁 Bucle de entrenamiento
  for epoch in range(num_epochs):
      L = 0.0       # pérdida acumulada
      N = 0         # número de ejemplos
      Acc = 0.0     # precisión acumulada en entrenamiento
      TestAcc = 0.0 # precisión en test
      TestN = 0     # cantidad de ejemplos de test

      # 🔹 Entrenamiento por lotes
      for X, y in train_iter:
          X, y = X.to(device), y.to(device)
          l = loss(model(X), y)           # calcula la pérdida
          trainer.zero_grad()             # resetea gradientes
          l.mean().backward()             # retropropagación
          trainer.step()                  # actualiza los pesos
          L += l.sum()
          N += l.numel()
          Acc += accuracy(model(X), y)    # mide precisión

      # 🔹 Evaluación en test
      for X, y in test_iter:
          X, y = X.to(device), y.to(device)
          TestN += y.numel()
          TestAcc += accuracy(model(X), y)

      # 🔹 Muestra resultados de la época
      print(f'epoch {epoch + 1}, loss {(L/N):f}, '
            f'train accuracy {(Acc/N):f}, test accuracy {(TestAcc/TestN):f}')


In [21]:
lr, num_epochs = 0.1, 10
img_size, patch_size = 96, 16
num_hiddens, mlp_num_hiddens, num_heads, num_blks = 512, 2048, 8, 2
emb_dropout, blk_dropout = 0.1, 0.1
model = ViT(img_size, patch_size, num_hiddens, mlp_num_hiddens, num_heads,
            num_blks, emb_dropout, blk_dropout, device, lr )

train_FashionMNIST_classifier(model,lr,num_epochs,resize=(img_size, img_size))


epoch 1, loss 0.857200            , train accuracy  0.689433, test accuracy 0.743600
epoch 2, loss 0.693485            , train accuracy  0.751383, test accuracy 0.733300
epoch 3, loss 0.722398            , train accuracy  0.740333, test accuracy 0.723200
epoch 4, loss 0.701535            , train accuracy  0.742817, test accuracy 0.741200
epoch 5, loss 0.686331            , train accuracy  0.748350, test accuracy 0.728700
epoch 6, loss 0.672452            , train accuracy  0.755717, test accuracy 0.688900
epoch 7, loss 0.695998            , train accuracy  0.745533, test accuracy 0.717400
epoch 8, loss 0.714233            , train accuracy  0.739833, test accuracy 0.741900
epoch 9, loss 0.737753            , train accuracy  0.716383, test accuracy 0.686000
epoch 10, loss 0.753620            , train accuracy  0.717333, test accuracy 0.707400


Puede notar que para datasets pequeños como Fashion-MNIST, nuestro transformer de visión implementado no supera a ResNet. Se pueden realizar observaciones similares incluso en el conjunto de datos de ImageNet (1,2 millones de imágenes). Esto se debe a que los transformers carecen de esos principios útiles en la convolución, como la localidad y la invariancia a la traslación. Sin embargo, el panorama cambia cuando se entrenan modelos más grandes en datasets más grandes (por ejemplo, 300 millones de imágenes), donde los transformers de visión superan a las ResNets por un amplio margen en la clasificación de imágenes, lo que demuestra la superioridad intrínseca de los transformers en escalabilidad.

